# Telecom Customer Churn – Exploratory Data Analysis

**Dataset:** IBM Telco Customer Churn (7,043 customers) enriched with synthetic billing history, usage, PEOTV subscription, and complaint data.  
**Goal:** Identify which customer segments churn most, quantify revenue at risk, and surface patterns for the modelling phase.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

In [ ]:
PROCESSED = Path('..') / 'data' / 'processed'
df = pd.read_csv(PROCESSED / 'dim_customer.csv')
fact_billing = pd.read_csv(PROCESSED / 'fact_billing.csv')
dim_date = pd.read_csv(PROCESSED / 'dim_date.csv')
print(f'Customers: {len(df):,}  |  Churn rate: {df["Churn"].mean():.1%}')
df.head()

## 1. Overall Churn Distribution

In [ ]:
from eda_utils import churn_distribution_pie, run_all_eda
churn_distribution_pie(df)
plt.show()

print(f"Churned: {df['Churn'].sum():,} ({df['Churn'].mean():.1%})")
print(f"Retained: {(df['Churn']==0).sum():,} ({(df['Churn']==0).mean():.1%})")

## 2. Churn Rate by Contract Type

**Key insight:** Month-to-month customers churn at ~43%, more than 4× the rate of two-year contract customers (~3%). This is the strongest single predictor and drives the main retention strategy.

In [ ]:
from eda_utils import churn_rate_by
rates_contract = churn_rate_by(df, 'Contract', 'Contract Type', 'churn_by_contract.png')
plt.show()
rates_contract

## 3. Churn Rate by Tenure

**Key insight:** New customers (0–12 months) churn at ~47%. After 3 years, churn drops below 10%. Early retention interventions (months 1–12) have the highest ROI.

In [ ]:
from eda_utils import churn_by_tenure
rates_tenure = churn_by_tenure(df)
plt.show()
rates_tenure

## 4. Churn Rate by Payment Method

In [ ]:
rates_payment = churn_rate_by(df, 'PaymentMethod', 'Payment Method', 'churn_by_payment.png', rotate=30)
plt.show()
rates_payment

## 5. Churn Rate by Internet Service

**Key insight:** Fiber optic customers churn at ~42% — surprising given it's the premium service. Suggests pricing dissatisfaction or unmet quality expectations.

In [ ]:
rates_internet = churn_rate_by(df, 'InternetService', 'Internet Service', 'churn_by_internet.png')
plt.show()
rates_internet

## 6. Churn Rate by Region

In [ ]:
rates_region = churn_rate_by(df, 'region', 'Region', 'churn_by_region.png', rotate=30)
plt.show()
rates_region

## 7. Revenue at Risk

Revenue at risk = sum of MonthlyCharges for customers who churned.  
This is the **monthly revenue the company has already lost** — and the minimum value of a successful retention programme.

In [ ]:
from eda_utils import revenue_at_risk
risk_df = revenue_at_risk(df)
plt.show()

total = risk_df['at_risk_revenue'].sum()
total_rev = df['MonthlyCharges'].sum()
print(f"Total monthly revenue: ${total_rev:,.0f}")
print(f"Revenue at risk (churned customers): ${total_rev * df['Churn'].mean():,.0f}")
print(f"That's {df['Churn'].mean():.1%} of monthly revenue.")

## 8. Monthly Revenue Trend (Synthetic History)

In [ ]:
from eda_utils import monthly_revenue_trend
monthly = monthly_revenue_trend(fact_billing, dim_date)
plt.show()
monthly.tail(6)

## 9. Complaint Tickets vs Churn

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
df.groupby('Churn')['complaint_tickets'].mean().plot.bar(
    color=['#2196F3', '#F44336'], ax=ax, edgecolor='white'
)
ax.set_xticklabels(['Retained', 'Churned'], rotation=0)
ax.set_ylabel('Avg Complaint Tickets')
ax.set_title('Average Complaint Tickets: Retained vs Churned')
ax.spines[['top','right']].set_visible(False)
for bar in ax.patches:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
            f'{bar.get_height():.2f}', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('../reports/figures/complaints_vs_churn.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

| Segment | Churn Rate | Action |
|---|---|---|
| Month-to-month contract | ~43% | Priority retention offer |
| Tenure 0–12 months | ~47% | Onboarding loyalty programme |
| Fiber optic internet | ~42% | Service quality review + price review |
| Electronic check payment | ~45% | Nudge to auto-pay (reduces friction) |

Proceed to `02_churn_model.ipynb` for predictive modelling.